# SIE Quickstart

Three functions - `encode`, `score`, `extract` - running on a free Colab GPU.

This notebook installs the SIE server, starts it in the background, and walks through the full API.

## 1. Install and start the server

Colab already has PyTorch, so this only installs the incremental dependencies (~1-2 min).

> **Note:** SIE requires Python 3.12. If your Colab runtime uses an older version, go to Runtime > Change runtime type and select Python 3.12.

In [1]:
!pip install -q sie-server sie-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 696.4/696.4 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.6/522.6 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import subprocess, time, urllib.request

# Start the server in the background
proc = subprocess.Popen(
    ["sie-server", "serve", "--host", "0.0.0.0", "--port", "8080"],
    stdout=open("/tmp/sie-server.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for the server to be ready
for i in range(60):
    try:
        urllib.request.urlopen("http://localhost:8080/healthz")
        print(f"Server ready (took ~{i}s)")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start - check /tmp/sie-server.log")

Server did not start - check /tmp/sie-server.log


## 2. Encode

Generate dense embeddings with a 400M-parameter model. The model downloads on first use (~1-2 min), then subsequent calls are fast.

In [ ]:
from sie_sdk import SIEClient
from sie_sdk.types import Item

client = SIEClient("http://localhost:8080")

result = client.encode("NovaSearch/stella_en_400M_v5", Item(text="Hello world"))
print(f"Shape: {result['dense'].shape}")   # (1024,)
print(f"First 5 values: {result['dense'][:5]}")

## 3. Score

Rerank documents by relevance using a cross-encoder.

In [ ]:
scores = client.score(
    "BAAI/bge-reranker-v2-m3",
    Item(text="What is machine learning?"),
    [Item(text="ML learns from data."), Item(text="The weather is sunny.")]
)

for s in scores["scores"]:
    print(f"  rank {s['rank']}: {s['score']:.3f} - {s['item_id']}")

## 4. Extract

Zero-shot named entity recognition - no training data needed.

In [ ]:
result = client.extract(
    "urchade/gliner_multi-v2.1",
    Item(text="Tim Cook is the CEO of Apple."),
    labels=["person", "organization"]
)

for e in result["entities"]:
    print(f"  {e['label']}: {e['text']} ({e['score']:.2f})")

## Next steps

- [API reference](https://superlinked.com/docs/reference/sdk) - full SDK documentation
- [All models](https://superlinked.com/models) - 85+ supported models
- [Deployment guide](https://superlinked.com/docs/deployment/docker) - run in production with Docker or Kubernetes
- [GitHub](https://github.com/superlinked/sie) - star the repo if this was useful